# 02 - Calidad y limpieza

**Objetivo del proyecto:** Analizar cómo se relaciona el tiempo mensual de visualización con el tipo de plan de suscripción y el país de los usuarios, para identificar patrones de consumo en la plataforma de streaming.

**Propósito de esta etapa:** aplicar decisiones de limpieza justificadas con la evidencia reunida en `01_inspeccion_inicial.ipynb`. Para cada decisión: **evidencia → acción → impacto**. El dataset original en `data/raw/` no se modifica; el resultado se guarda en `data/processed/`. Cada transformación se registra en `logs/pipeline_log.csv`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_json('../data/raw/streaming_users_dirty.json')
n_inicial = len(df)
log = []  # cada elemento: (paso, descripcion, filas, nulos, retencion_pct)

def registrar_paso(paso, descripcion, df_actual):
    filas = len(df_actual)
    nulos = int(df_actual.isna().sum().sum())
    retencion = round(filas / n_inicial * 100, 2)
    log.append({'Paso': paso, 'Descripción': descripcion, 'Filas': filas, 'Nulos': nulos, 'Retención (%)': retencion})

registrar_paso(0, 'Estado inicial del dataset (data/raw)', df)
df.shape

(8160, 8)

## 1. Duplicados

**Evidencia (notebook 01):** 160 `user_id` duplicados, 126 filas 100% duplicadas.

**Decisión:** eliminar filas completamente duplicadas (son claramente el mismo registro repetido). Para los `user_id` duplicados que *no* son filas idénticas, se conserva el primer registro por `user_id`, asumiendo que representa la carga original y el resto son reinserciones erróneas — no hay forma de saber cuál es "correcto" sin más contexto, por lo que quedarse con el primero es la decisión más conservadora.

In [2]:
df = df.drop_duplicates()
registrar_paso(1, 'Eliminación de filas 100% duplicadas', df)

df = df.drop_duplicates(subset='user_id', keep='first')
registrar_paso(2, 'Eliminación de user_id duplicados (se conserva el primero)', df)

print(f"Filas restantes: {len(df)}")

Filas restantes: 8000


## 2. Normalización de variables categóricas

**Evidencia:** `subscription_plan`, `country` y `favorite_genre` tienen decenas de variantes de texto (mayúsculas, espacios, tildes, abreviaturas, anglicismos) para un número mucho menor de categorías reales.

**Decisión:** normalizar texto (trim + minúsculas + sin tildes) y mapear variantes conocidas a la categoría canónica.

In [3]:
import unicodedata

def normalizar_texto(s):
    if pd.isna(s):
        return s
    s = str(s).strip().lower()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return s

mapa_plan = {
    'basico': 'Básico', 'basic': 'Básico',
    'estandar': 'Estándar', 'std': 'Estándar', 'standard': 'Estándar',
    'premium': 'Premium', 'premiun': 'Premium',
}

df['subscription_plan'] = df['subscription_plan'].apply(normalizar_texto).map(mapa_plan)
registrar_paso(3, 'Normalización de subscription_plan (15 variantes -> 3 categorías)', df)
df['subscription_plan'].value_counts(dropna=False)

subscription_plan
Básico      3600
Estándar    2817
Premium     1583
Name: count, dtype: int64

In [4]:
mapa_pais = {
    'brasil': 'Brasil', 'brazil': 'Brasil',
    'chile': 'Chile',
    'mexico': 'México',
    'uruguay': 'Uruguay',
    'peru': 'Perú',
    'colombia': 'Colombia',
    'argentina': 'Argentina',
    'bra': 'Brasil', 'chl': 'Chile', 'mex': 'México',
    'ury': 'Uruguay', 'per': 'Perú', 'col': 'Colombia', 'arg': 'Argentina',
}

df['country'] = df['country'].apply(normalizar_texto).map(mapa_pais)
registrar_paso(4, 'Normalización de country (26 variantes -> 7 países)', df)
df['country'].value_counts(dropna=False)

country
Chile        1164
Brasil       1156
México       1156
Uruguay      1143
Colombia     1142
Perú         1134
Argentina    1105
Name: count, dtype: int64

In [5]:
mapa_genero = {
    'comedia': 'Comedia', 'comedy': 'Comedia',
    'drama': 'Drama',
    'documental': 'Documental', 'documentary': 'Documental', 'doc': 'Documental',
    'thriller': 'Thriller', 'thriler': 'Thriller',
    'romance': 'Romance',
    'accion': 'Acción', 'action': 'Acción',
    'crime': 'Crime', 'crimen': 'Crime',
}

df['favorite_genre'] = df['favorite_genre'].apply(normalizar_texto).map(mapa_genero)
registrar_paso(5, 'Normalización de favorite_genre (29 variantes -> 7 géneros)', df)
df['favorite_genre'].value_counts(dropna=False)

favorite_genre
Comedia       1137
Drama         1115
Romance       1110
Documental    1107
Thriller      1104
Acción        1103
Crime         1084
NaN            240
Name: count, dtype: int64

**Nulos en `favorite_genre`:** no es una variable central del objetivo del proyecto (que se centra en tiempo de visualización, plan y país). Se decide conservar los nulos como categoría `"No especificado"` en vez de imputar o eliminar filas, para no perder información de las variables centrales de esas filas.

In [6]:
df['favorite_genre'] = df['favorite_genre'].fillna('No especificado')
registrar_paso(6, "Nulos de favorite_genre asignados a categoría 'No especificado'", df)

## 3. `age`

**Evidencia:** rango de -5 a 150. Una plataforma de streaming asume usuarios entre, razonablemente, 13 y 100 años (fuera de ese rango son errores de carga, no edades reales).

**Decisión:** los valores fuera de [13, 100] se consideran inválidos y se reemplazan por nulo, y luego se imputan con la mediana (medida robusta a outliers) agrupada por `subscription_plan`, para no perder filas de una variable que no es la central de este análisis.

In [7]:
fuera_de_rango = ((df['age'] < 13) | (df['age'] > 100)).sum()
print(f"Valores de age fuera de rango [13, 100]: {fuera_de_rango}")

df.loc[(df['age'] < 13) | (df['age'] > 100), 'age'] = np.nan
df['age'] = df.groupby('subscription_plan')['age'].transform(lambda s: s.fillna(s.median()))
df['age'] = df['age'].round().astype(int)
registrar_paso(7, 'age fuera de [13,100] -> nulo -> imputado con mediana por plan', df)
df['age'].describe()

Valores de age fuera de rango [13, 100]: 120


count    8000.000000
mean       33.707875
std        11.455618
min        13.000000
25%        26.000000
50%        33.000000
75%        41.000000
max        80.000000
Name: age, dtype: float64

## 4. `monthly_watch_time_mins` (variable central del objetivo)

**Evidencia:** valores `NaN` (193), valores negativos (49, imposibles) y un grupo de 20 registros en exactamente 99999.0 (valor centinela).

**Decisión:** por ser la variable central del análisis, se trata con cuidado especial:
- Negativos y el centinela 99999.0 se consideran errores de carga -> se pasan a nulo (no se puede saber el valor real).
- Los `NaN` resultantes se imputan con la **mediana por combinación de `subscription_plan` y `country`**, ya que precisamente esperamos que el tiempo de visualización varíe según esas dos variables — usar la mediana global sería introducir un sesgo que aplanaría el patrón que queremos analizar.

In [8]:
invalidos = ((df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] == 99999.0)).sum()
print(f"Valores inválidos (negativos o centinela 99999.0): {invalidos}")

df.loc[(df['monthly_watch_time_mins'] < 0) | (df['monthly_watch_time_mins'] == 99999.0), 'monthly_watch_time_mins'] = np.nan
registrar_paso(8, 'monthly_watch_time_mins negativos/centinela -> nulo', df)

nulos_antes = df['monthly_watch_time_mins'].isna().sum()
df['monthly_watch_time_mins'] = df.groupby(['subscription_plan','country'])['monthly_watch_time_mins']\
    .transform(lambda s: s.fillna(s.median()))
# fallback por si alguna combinación quedó sin datos para calcular mediana
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].fillna(df['monthly_watch_time_mins'].median())
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].round(1)
registrar_paso(9, f'monthly_watch_time_mins: {nulos_antes} nulos imputados con mediana por (plan, país)', df)
df['monthly_watch_time_mins'].describe()

Valores inválidos (negativos o centinela 99999.0): 69


count     8000.000000
mean       868.501900
std       1887.948747
min          0.000000
25%        499.150000
50%        771.350000
75%       1060.300000
max      50000.000000
Name: monthly_watch_time_mins, dtype: float64

## 5. `customer_support_tickets`

**Evidencia:** valores negativos (-1) y un grupo en 150 (fuera de la distribución típica).

**Decisión:** no es variable central del objetivo. Se acota a un rango plausible [0, 30] (el percentil 99 real del resto de la distribución está muy por debajo de eso); los valores fuera de rango se llevan al límite más cercano (*capping*), preservando la fila sin inventar un valor específico.

In [9]:
print(df['customer_support_tickets'].quantile([0.5, 0.9, 0.95, 0.99]))

df['customer_support_tickets'] = df['customer_support_tickets'].clip(lower=0, upper=30)
registrar_paso(10, 'customer_support_tickets acotado al rango [0, 30] (capping)', df)

0.50    1.0
0.90    2.0
0.95    3.0
0.99    5.0
Name: customer_support_tickets, dtype: float64


## 6. `last_login_date`

**Evidencia:** 3+ formatos de fecha distintos, nulos, y strings inválidos (`"not_available"`, `"0000-00-00"`).

**Decisión:** no es variable central del objetivo (que se enfoca en plan, país y tiempo de visualización), por lo que no se descartan filas por esto. Se parsea cada formato conocido a `datetime`, y lo que no se puede interpretar (nulos, `"not_available"`, `"0000-00-00"`) queda como `NaT` — se conserva la fila igual, ya que las demás columnas siguen siendo válidas.

In [10]:
def parsear_fecha(v):
    if pd.isna(v) or v in ('not_available', '0000-00-00'):
        return pd.NaT
    for fmt in ('%Y-%m-%d', '%m-%d-%Y', '%Y/%m/%d'):
        try:
            return pd.to_datetime(v, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['last_login_date'] = df['last_login_date'].apply(parsear_fecha)
registrar_paso(11, 'last_login_date parseada a datetime unificado; inválidos -> NaT', df)
print(f"Fechas válidas: {df['last_login_date'].notna().sum()} / {len(df)}")

Fechas válidas: 7544 / 8000


## 7. Verificación final y guardado

In [11]:
df.info()
print()
print("Nulos por columna:")
print(df.isna().sum())

<class 'pandas.DataFrame'>
Index: 8000 entries, 0 to 7999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   user_id                   8000 non-null   int64         
 1   age                       8000 non-null   int64         
 2   subscription_plan         8000 non-null   str           
 3   monthly_watch_time_mins   8000 non-null   float64       
 4   country                   8000 non-null   str           
 5   favorite_genre            8000 non-null   str           
 6   last_login_date           7544 non-null   datetime64[us]
 7   customer_support_tickets  8000 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 733.6 KB

Nulos por columna:
user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins       0
country                       0
favorite_genre                0

In [12]:
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print(f"Dataset procesado guardado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"Retención respecto al original: {round(len(df)/n_inicial*100, 2)}%")

Dataset procesado guardado: 8000 filas x 8 columnas
Retención respecto al original: 98.04%


In [13]:
log_df = pd.DataFrame(log)
log_df.to_csv('../logs/pipeline_log.csv', index=False)
log_df

,Paso,Descripción,Filas,Nulos,Retención (%)
0,0,Estado inicial del dataset (data/raw),8160,753,100.00
1,1,Eliminación de filas 100% duplicadas,8034,753,98.46
2,2,Eliminación de user_id duplicados (se conserva...,8000,753,98.04
3,3,Normalización de subscription_plan (15 variant...,8000,753,98.04
4,4,Normalización de country (26 variantes -> 7 pa...,8000,753,98.04
5,5,Normalización de favorite_genre (29 variantes ...,8000,753,98.04
6,6,Nulos de favorite_genre asignados a categoría ...,8000,513,98.04
7,7,"age fuera de [13,100] -> nulo -> imputado con ...",8000,513,98.04
8,8,monthly_watch_time_mins negativos/centinela ->...,8000,582,98.04
9,9,monthly_watch_time_mins: 262 nulos imputados c...,8000,320,98.04


## 8. Resumen del impacto

Se partió de 8160 filas con múltiples problemas de calidad (duplicados, valores imposibles, formatos inconsistentes, y datos faltantes o centinela en la variable central del análisis). Después de la limpieza se conservó la gran mayoría de los registros (ver `logs/pipeline_log.csv` para el detalle fila por fila de cada paso), priorizando siempre no perder información en `monthly_watch_time_mins`, `subscription_plan` y `country` -- las tres variables que responden directamente al objetivo del proyecto. El dataset resultante queda listo para el análisis exploratorio en `03_eda.ipynb`.